In [ ]:
import pandas as pd

In [ ]:
headlines= pd.read_csv("merged_headlines_dataset_sorted.csv")

In [ ]:
headlines["published"].max()
headlines

,source,title,published
0,zdnet.com,US wiped some hard drives of Russia's 'troll f...,2019-03-01
1,nypost.com,Alleged home invader claims he was just chill...,2019-03-01
2,dailymail.co.uk,Minneapolis cop who killed Australian woman 'p...,2019-03-01
3,mobile.twitter.com,": Créelo o no, pero para el 2017, Nostradamus ...",2019-03-01
4,americanthinker.com,The Resistance: everything they accuse Trump o...,2019-03-01
...,...,...,...
22714,Google News,Starfish Necklace for Women Gold Star Swirl Pe...,2025-10-08
22715,Google News,"Black Pearl Necklace for Women, AAAA Quality G...",2025-10-08
22716,Google News,Round-Cut Moissanite Engagement Ring Set for W...,2025-10-08
22717,Google News,FindChic Handcuff Necklace Padlock Pendant Cus...,2025-10-08


In [ ]:
!pip install yfinance

In [ ]:
import yfinance as yf

gold = yf.download("GC=F", start="2019-03-01", end="2025-12-31", interval="1d")
print(len(gold))
gold.to_csv("daily_gold_prices.csv")


/tmp/ipython-input-2192772700.py:3: FutureWarning: YF.download() has changed argument auto_adjust default to True
  gold = yf.download("GC=F", start="2019-03-01", end="2025-12-31", interval="1d")
[*********************100%***********************]  1 of 1 completed

1666


In [ ]:
gold_dum = pd.read_csv("daily_gold_prices.csv")
gold

Price,Close,High,Low,Open,Volume
Ticker,GC=F,GC=F,GC=F,GC=F,GC=F
Date,,,,,
2019-03-01,1296.400024,1312.199951,1288.900024,1312.199951,110
2019-03-04,1284.800049,1287.000000,1281.900024,1285.500000,28
2019-03-05,1282.000000,1287.000000,1282.000000,1287.000000,114
2019-03-06,1284.900024,1287.099976,1282.599976,1287.099976,14
2019-03-07,1283.800049,1284.599976,1281.000000,1281.000000,11
...,...,...,...,...,...
2025-10-06,3948.500000,3960.000000,3926.800049,3931.300049,1267
2025-10-07,3976.600098,3981.500000,3940.000000,3959.399902,2903


In [ ]:

col_names = ["Date", "Close", "High", "Low", "Open", "Volume"]

gold = pd.read_csv("daily_gold_prices.csv", skiprows=2, names=col_names)
gold = gold.drop(index=0)
gold

# Convert Date to datetime
#gold["Date"] = pd.to_datetime(gold["Date"])


,Date,Close,High,Low,Open,Volume
1,2019-03-01,1296.400024,1312.199951,1288.900024,1312.199951,110.0
2,2019-03-04,1284.800049,1287.000000,1281.900024,1285.500000,28.0
3,2019-03-05,1282.000000,1287.000000,1282.000000,1287.000000,114.0
4,2019-03-06,1284.900024,1287.099976,1282.599976,1287.099976,14.0
5,2019-03-07,1283.800049,1284.599976,1281.000000,1281.000000,11.0
...,...,...,...,...,...,...
1662,2025-10-06,3948.500000,3960.000000,3926.800049,3931.300049,1267.0
1663,2025-10-07,3976.600098,3981.500000,3940.000000,3959.399902,2903.0
1664,2025-10-08,4043.300049,4049.199951,3987.199951,3987.199951,2179.0
1665,2025-10-09,3946.300049,4046.199951,3940.000000,4011.199951,3130.0


In [ ]:
headlines = headlines.sort_values(by='published', ascending=True)
headlines["published"]

,published
0,2019-03-01
22,2019-03-01
23,2019-03-01
24,2019-03-01
25,2019-03-01
...,...
22343,2025-10-08
22344,2025-10-08
22345,2025-10-08
22339,2025-10-08


In [ ]:
!pip install transformers

In [ ]:
!pip install torch

In [ ]:
!pip install huggingface_hub

In [ ]:
import requests
import pandas as pd
import time
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scipy.special import softmax
import numpy as np
import torch
import huggingface_hub

In [ ]:
model_name = "ProsusAI/finbert"
huggingface_hub.constants.HF_HUB_HTTP_TIMEOUT = 60  # extend timeout


In [ ]:
for attempt in range(5):
    try:
        print(f"\nAttempt {attempt+1}/5: Loading FinBERT model...")
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForSequenceClassification.from_pretrained(model_name)
        print("✅ FinBERT loaded successfully!")
        break
    except Exception as e:
        print("⚠️ Error:", e)
        if attempt < 4:
            print("⏳ Retrying in 5 seconds...\n")
            time.sleep(5)
        else:
            raise e



Attempt 1/5: Loading FinBERT model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

✅ FinBERT loaded successfully!


In [24]:
sentiments = []

for text in tqdm(headlines["title"].fillna(""), desc="Analyzing Sentiment"):
    try:
        inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
        outputs = model(**inputs)
        scores = softmax(outputs.logits.detach().numpy()[0])
        sentiments.append({
            "negative": float(scores[0]),
            "neutral": float(scores[1]),
            "positive": float(scores[2])
        })
    except Exception as e:
        sentiments.append({"negative": np.nan, "neutral": np.nan, "positive": np.nan})
        print("⚠️ Error on text:", text, "|", e)

sent_df = pd.DataFrame(sentiments)
headline_df = pd.concat([headlines.reset_index(drop=True), sent_df], axis=1)


Analyzing Sentiment: 100%|██████████| 22719/22719 [51:22<00:00,  7.37it/s]


In [30]:
import pandas as pd

# 1. Load gold prices CSV
col_names = ["Date", "Close", "High", "Low", "Open", "Volume"]
gold = pd.read_csv("daily_gold_prices.csv", skiprows=2, names=col_names)

# Drop any unnecessary first row if needed
gold = gold.drop(index=0)

# Convert Date to datetime (MM/DD/YYYY)
gold['Date'] = pd.to_datetime(gold['Date'], dayfirst=False, errors='coerce')



# Rename date column to match gold CSV
headline_df.rename(columns={"published": "Date"}, inplace=True)

# Convert Date to datetime (MM/DD/YYYY)
headline_df['Date'] = pd.to_datetime(headline_df['Date'], dayfirst=False, errors='coerce')

# 3. Merge datasets
merged = pd.merge(gold, headline_df, on='Date', how='outer')

# 4. Sort by Date ascending-
merged = merged.sort_values('Date').reset_index(drop=True)



In [29]:
merged.to_csv("gold_sentiment_merged.csv", index=False)

In [ ]:

!pip install pandas_datareader --quiet

import pandas_datareader.data as web
import pandas as pd
import datetime


start = datetime.datetime(2019, 3, 1)
end = datetime.datetime(2025, 10, 10)


interest_rate = web.DataReader("FEDFUNDS", "fred", start, end)


inflation = web.DataReader("FPCPITOTLZGUSA", "fred", start, end)


usd_index = web.DataReader("DTWEXBGS", "fred", start, end)


macro_df = pd.concat([interest_rate, inflation, usd_index], axis=1)
macro_df.columns = ["interest_rate", "inflation", "usd_index"]


macro_df = macro_df.reset_index()
macro_df.rename(columns={"index": "date"}, inplace=True)


macro_df.to_csv("macro_data.csv", index=False)

print("✅ macro_data.csv created successfully!")
print(macro_df.head())


✅ macro_data.csv created successfully!
        DATE  interest_rate  inflation  usd_index
0 2019-03-01           2.41        NaN   114.5350
1 2019-03-04            NaN        NaN   114.6903
2 2019-03-05            NaN        NaN   114.8019
3 2019-03-06            NaN        NaN   114.9697
4 2019-03-07            NaN        NaN   115.4905


In [ ]:
import pandas as pd


gold_df = pd.read_csv("gold_sentiment_merged.csv")
macro_df = pd.read_csv("macro_data.csv")


gold_df['Date'] = pd.to_datetime(gold_df['Date'])
macro_df.rename(columns={'DATE': 'Date'}, inplace=True)
macro_df['Date'] = pd.to_datetime(macro_df['Date'])


merged_df = pd.merge(gold_df, macro_df, on='Date', how='left')


merged_df[['interest_rate', 'inflation', 'usd_index']] = (
    merged_df[['interest_rate', 'inflation', 'usd_index']].ffill().bfill()
)


merged_df.to_csv("gold_macro_sentiment.csv", index=False)


print("Rows:", merged_df.shape[0])
print(merged_df.head())


✅ gold_macro_sentiment.csv created successfully!
Rows: 23693
        Date        Close         High          Low         Open  Volume  \
0 2019-03-01  1296.400024  1312.199951  1288.900024  1312.199951   110.0   
1 2019-03-01  1296.400024  1312.199951  1288.900024  1312.199951   110.0   
2 2019-03-01  1296.400024  1312.199951  1288.900024  1312.199951   110.0   
3 2019-03-01  1296.400024  1312.199951  1288.900024  1312.199951   110.0   
4 2019-03-01  1296.400024  1312.199951  1288.900024  1312.199951   110.0   

               source                                              title  \
0           zdnet.com  US wiped some hard drives of Russia's 'troll f...   
1          nypost.com  Alleged home invader claims he was just chill...   
2     dailymail.co.uk  Minneapolis cop who killed Australian woman 'p...   
3  mobile.twitter.com  : Créelo o no, pero para el 2017, Nostradamus ...   
4         reuters.com  U.S. companies put record number of robots to ...   

   negative   neutral  po